In [ ]:
# BLOQUE 1 — Parámetros del snapshot GTFS a cargar en Bronze

import re

# Localizo los módulos en el repositorio completo, igual que el notebook Silver.
from pathlib import Path
import sys

dbutils.widgets.text("repo_root", "", "Raíz del repositorio en Workspace")
root_parameter = dbutils.widgets.get("repo_root").strip()
if root_parameter and not Path(root_parameter).is_absolute():
    raise ValueError("repo_root debe ser una ruta absoluta de Workspace.")
current = Path.cwd().resolve()
roots = [Path(root_parameter)] if root_parameter else [current, *current.parents]
project_root = next((p for p in roots if (p / "databricks/gtfs_bronze/runtime.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Sincroniza el repositorio completo y configura repo_root; falta gtfs_bronze.")
module_path = str(project_root / "databricks")
if module_path not in sys.path:
    sys.path.insert(0, module_path)

# Estos defaults corresponden a mi entorno actual; el Job puede reemplazarlos.
for key, default in {"catalog": "workspace", "schema": "gtfs_bogota"}.items():
    dbutils.widgets.text(key, default)
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
from gtfs_bronze.contracts import quoted_table
quoted_table(f"{catalog}.{schema}.configuration_check")
spark.conf.set("spark.sql.session.timeZone", "UTC")

dbutils.widgets.text("staging_volume", "staging_gtfs")
staging_volume = dbutils.widgets.get("staging_volume").strip()
quoted_table(f"{catalog}.{schema}.{staging_volume}")
staging_volume_path = f"/Volumes/{catalog}/{schema}/{staging_volume}"

# La procedencia llega desde Raw como versión del proveedor y hash inmutable.
dbutils.widgets.text("feed_version", "2025-10-28", "Versión GTFS")
dbutils.widgets.text(
    "snapshot_sha256",
    "07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952",
    "SHA-256 del snapshot",
)
feed_version = dbutils.widgets.get("feed_version").strip()
snapshot_sha256 = dbutils.widgets.get("snapshot_sha256").strip().lower()

if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", feed_version):
    raise ValueError("feed_version debe tener el formato AAAA-MM-DD.")
if not re.fullmatch(r"[0-9a-f]{64}", snapshot_sha256):
    raise ValueError("snapshot_sha256 debe ser un hash SHA-256 de 64 caracteres.")

staging_snapshot_path = (
    f"{staging_volume_path}/gtfs_static/"
    f"feed_version={feed_version}/"
    f"sha256={snapshot_sha256}"
)

print(f"Snapshot de Staging: {staging_snapshot_path}")
display(dbutils.fs.ls(staging_snapshot_path))
dbutils.widgets.text("show_diagnostics", "false")
show_diagnostics_parameter = dbutils.widgets.get("show_diagnostics").strip().lower()
if show_diagnostics_parameter not in {"true", "false"}:
    raise ValueError("show_diagnostics debe ser true o false.")
show_diagnostics = show_diagnostics_parameter == "true"


In [0]:
# BLOQUE 2 — Contrato mínimo de las entidades GTFS de referencia

# Los calendarios se validan como grupo: debe existir al menos uno.
# Los demás requisitos corresponden al alcance actual del feed de Bogotá.
# Las columnas opcionales también se preservan como texto en Bronze.
reference_entities = [
    {
        "file_name": "agency.txt",
        "table_name": "bronze_gtfs_agency",
        "is_required": True,
        "required_columns": ["agency_name", "agency_url", "agency_timezone"],
    },
    {
        "file_name": "stops.txt",
        "table_name": "bronze_gtfs_stops",
        "is_required": True,
        "required_columns": ["stop_id", "stop_name", "stop_lat", "stop_lon"],
    },
    {
        "file_name": "trips.txt",
        "table_name": "bronze_gtfs_trips",
        "is_required": True,
        "required_columns": ["route_id", "service_id", "trip_id"],
    },
    {
        "file_name": "calendar.txt",
        "table_name": "bronze_gtfs_calendar",
        "is_required": False,
        "required_columns": [
            "service_id", "monday", "tuesday", "wednesday", "thursday",
            "friday", "saturday", "sunday", "start_date", "end_date",
        ],
    },
    {
        "file_name": "calendar_dates.txt",
        "table_name": "bronze_gtfs_calendar_dates",
        "is_required": False,
        "required_columns": ["service_id", "date", "exception_type"],
    },
    {
        "file_name": "frequencies.txt",
        "table_name": "bronze_gtfs_frequencies",
        "is_required": False,
        "required_columns": ["trip_id", "start_time", "end_time", "headway_secs"],
    },
    {
        "file_name": "fare_attributes.txt",
        "table_name": "bronze_gtfs_fare_attributes",
        "is_required": False,
        "required_columns": ["fare_id", "price", "currency_type", "payment_method", "transfers"],
    },
    {
        "file_name": "fare_rules.txt",
        "table_name": "bronze_gtfs_fare_rules",
        "is_required": False,
        "required_columns": ["fare_id"],
    },
    {
        "file_name": "feed_info.txt",
        "table_name": "bronze_gtfs_feed_info",
        "is_required": False,
        "required_columns": [
            "feed_publisher_name",
            "feed_publisher_url",
            "feed_lang",
        ],
    },
]

for entity in reference_entities:
    print(f"{entity['file_name']} -> {catalog}.{schema}.{entity['table_name']}")

In [ ]:
# BLOQUE 3 — Funciones compartidas de validación y carga Bronze

from gtfs_bronze.runtime import read_gtfs_header, source_frame, load_frame
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType, LongType, StringType, StructField, StructType


def load_entity_to_bronze(entity: dict) -> tuple:
    """Conserva la fuente como texto y rechaza cargas parciales sin duplicar filas."""
    source_path = f"{staging_snapshot_path}/{entity['file_name']}"
    table_name = f"{catalog}.{schema}.{entity['table_name']}"
    source_columns = read_gtfs_header(source_path)
    missing_columns = sorted(set(entity['required_columns']) - set(source_columns))
    if missing_columns:
        raise RuntimeError(f"Contrato GTFS incumplido en {entity['file_name']}. Faltan columnas: {', '.join(missing_columns)}")
    count, reused = load_frame(
        spark, source_frame(spark, source_path, source_columns), table=table_name,
        source_path=source_path, feed_version=feed_version, sha256=snapshot_sha256,
    )
    return entity['file_name'], table_name, count, reused


load_result_schema = StructType([
    StructField("source_file_name", StringType(), False),
    StructField("bronze_table_name", StringType(), False),
    StructField("source_row_count", LongType(), False),
    StructField("snapshot_already_loaded", BooleanType(), False),
])


In [ ]:
# Diagnóstico opcional; notebook 07 mantiene la compuerta obligatoria.
if show_diagnostics:
    # BLOQUE 3A — Diagnóstico de compatibilidad de esquema antes de Bronze

    from pyspark.sql import functions as F
    from pyspark.sql.types import (
        BooleanType,
        LongType,
        StringType,
        StructField,
        StructType,
    )

    # Estas columnas son creadas por el pipeline y no pertenecen al archivo GTFS.
    technical_columns = {
        "source_feed_version",
        "source_sha256",
        "source_file_path",
        "bronze_loaded_at_utc",
    }

    # Revisamos cada archivo disponible contra la tabla Delta histórica equivalente.
    # También verificamos si el snapshot 2026 alcanzó a cargarse antes del error.
    available_staging_files = {
        file.name
        for file in dbutils.fs.ls(staging_snapshot_path)
    }

    schema_audit_rows = []

    for entity in reference_entities:
        if entity["file_name"] not in available_staging_files:
            continue

        source_path = f"{staging_snapshot_path}/{entity['file_name']}"
        table_name = f"{catalog}.{schema}.{entity['table_name']}"

        source_columns = set(read_gtfs_header(source_path))
        incoming_columns = source_columns | technical_columns
        table_exists = spark.catalog.tableExists(table_name)

        if table_exists:
            target_columns = set(spark.table(table_name).columns)

            rows_for_snapshot = (
                spark.table(table_name)
                .filter(F.col("source_sha256") == snapshot_sha256)
                .count()
            )

            new_in_source = sorted(incoming_columns - target_columns)
            absent_in_source = sorted(target_columns - incoming_columns)
        else:
            rows_for_snapshot = 0
            new_in_source = sorted(incoming_columns)
            absent_in_source = []

        schema_audit_rows.append((
            entity["file_name"],
            table_name,
            table_exists,
            rows_for_snapshot,
            ", ".join(new_in_source) or "ninguna",
            ", ".join(absent_in_source) or "ninguna",
        ))

    schema_audit_schema = StructType([
        StructField("source_file_name", StringType(), False),
        StructField("bronze_table_name", StringType(), False),
        StructField("table_already_exists", BooleanType(), False),
        StructField("rows_for_current_snapshot", LongType(), False),
        StructField("new_columns_vs_table", StringType(), False),
        StructField("historic_columns_missing_in_source", StringType(), False),
    ])

    display(
        spark.createDataFrame(
            schema_audit_rows,
            schema=schema_audit_schema,
        )
        .orderBy("source_file_name")
    )


In [0]:
# BLOQUE 4 — Carga idempotente de entidades de referencia en Delta Bronze

# GTFS permite archivos opcionales. Registramos su ausencia sin detener
# el pipeline ni inventar filas para una versión que no los publicó.
available_staging_files = {
    file.name
    for file in dbutils.fs.ls(staging_snapshot_path)
}

# Su ausencia individual es válida; la ausencia del grupo impide la carga.
if not ({"calendar.txt", "calendar_dates.txt"} & available_staging_files):
    raise RuntimeError(
        "Falta la definición de servicio: publica calendar.txt, "
        "calendar_dates.txt o ambos antes de cargar Bronze."
    )

missing_required_files = [
    entity["file_name"]
    for entity in reference_entities
    if entity["is_required"]
    and entity["file_name"] not in available_staging_files
]
if missing_required_files:
    raise RuntimeError(
        "Faltan archivos GTFS requeridos: "
        + ", ".join(missing_required_files)
    )

entities_to_load = [
    entity
    for entity in reference_entities
    if entity["file_name"] in available_staging_files
]
skipped_optional_files = [
    entity["file_name"]
    for entity in reference_entities
    if not entity["is_required"]
    and entity["file_name"] not in available_staging_files
]

load_results = [
    load_entity_to_bronze(entity)
    for entity in entities_to_load
]

load_results_df = spark.createDataFrame(
    load_results,
    schema=load_result_schema,
)

print("Archivos opcionales no publicados: " + str(skipped_optional_files))
display(load_results_df.orderBy("source_file_name"))

In [ ]:
# Diagnóstico opcional; notebook 07 mantiene la compuerta obligatoria.
if show_diagnostics:
    # BLOQUE 5 — Verificación inicial de paradas cargadas

    bronze_stops_table = f"{catalog}.{schema}.bronze_gtfs_stops"

    display(
        spark.table(bronze_stops_table)
        .filter(F.col("source_sha256") == snapshot_sha256)
        .orderBy("stop_id")
        .limit(20)
    )

    display(spark.sql(f"describe detail {bronze_stops_table}"))
